# RK T4 Rev 5.0 EVT — 09 — Master Cross-Domain FMEA, Provenance Anti-Spoofing Gate, HTML Dashboard & Rev 6 Patch Generator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sc8z35a-collab/GIT-Y5/blob/main/colab_ultimate_signoff/09_cross_fmea_master_signoff.ipynb)

| Field | Value |
|---|---|
| **Recommended Colab Pro Runtime** | `CPU (Standard)` |
| **Execution Wave (DAG)** | `Wave 3 (Final Signoff Gate — after 00..08 complete)` |
| **Estimated Duration** | `2–5 min` |
| **Source Repository** | `https://github.com/sc8z35a-collab/GIT-Y5` (`main`) |
| **Read-Only Guarantee** | All solver work runs in `/tmp/rk_t4_work/` (auto-GC'd per case) & persists atomically to Google Drive; tracked design files remain 100% unmodified. |

### Verification Scope
1. **Strict Schema & Solver Provenance Anti-Spoofing Gate**: validates all 10 upstream reports (`00`, `01`, `02`, `03`, `04a`, `04b`, `05`, `06`, `07`, `08`) against `common/schemas.py`, verifies identical `git_commit_sha` & `input_files_sha256`, rejects any dummy 1 mm cube cheat, and blocks tapeout if ANY notebook ran in `QUICK_CHECK` (`PRELIMINARY_QUICK_CHECK_ONLY`) or has `FAIL > 0` (`SIGNOFF_REJECTED_NEEDS_REV6_PATCH`).
2. **4 Mandatory Cross-Domain Physical Consistency Checks**:
   - NB 06 peak PCB temp $\rho(T)$ → NB 00/02 hot PDN IR drop
   - NB 04b reflow warpage + NB 08 3D tolerance + NB 06 3-year Si-C swelling combined Z clearance
   - NB 03 max RF PA dissipation ↔ NB 06 mezzanine thermal heat source
   - NB 04a peak drop shock G ↔ NB 07/08 OIS mechanical stop & DF40C/MHF4 retention limits
3. **Deliverables**: RPN-ranked DFMEA/PFMEA, **`rev6_patch_orders.json`**, single self-contained interactive **`RK_T4_MASTER_SIGNOFF_REPORT.html`**, and **`RK_T4_EVT_TAPEOUT_CERTIFICATE.md`** clearly separating digitally proven domains from irreducible real-hardware EVT domains.


In [ ]:
#@title 1. Standalone Repository Self-Clone, Google Drive Mount & Hardware Bootstrap
REPO_URL = "https://github.com/sc8z35a-collab/GIT-Y5.git" #@param {type:"string"}
REPO_BRANCH = "main" #@param {type:"string"}
RUN_MODE = "ULTRA_SIGNOFF" #@param ["QUICK_CHECK", "ULTRA_SIGNOFF"]
BATCH_INDEX = 0 #@param {type:"integer"}
RANDOM_SEED = 42 #@param {type:"integer"}
DRIVE_ROOT = "/content/drive/MyDrive/RK_T4_VirtualLab" #@param {type:"string"}

import os
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/rk_t4_repo")
if not (REPO_DIR / ".git").exists():
    clone_url = REPO_URL
    token = os.environ.get("GITHUB_TOKEN", "")
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN") or ""
        except Exception:
            token = ""
    if token and clone_url.startswith("https://github.com/"):
        clone_url = clone_url.replace("https://github.com/", f"https://{token}@github.com/", 1)
    res = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, clone_url, str(REPO_DIR)],
        text=True,
        capture_output=True,
    )
    if res.returncode != 0:
        safe_err = res.stderr.replace(token, "***") if token else res.stderr
        raise RuntimeError(f"Standalone git clone failed: {safe_err}")

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from colab_ultimate_signoff.common.colab_bootstrap import (
    bootstrap_notebook,
    ensure_cpu_cae_env,
    run_in_env,
    verify_repo_unmodified,
)
from colab_ultimate_signoff.common.schemas import validate_report_file

ctx = bootstrap_notebook(
    repo_url=REPO_URL,
    repo_branch=REPO_BRANCH,
    repo_dir=REPO_DIR,
    drive_root=DRIVE_ROOT,
    require_gpu=False,
)
print(f"[NB 09] Repo Commit : {ctx.git_commit_sha}")
print(f"[NB 09] Runtime HW  : OS={ctx.hw.os_codename}, CPU={ctx.hw.cpu_count} (OMP={ctx.hw.omp_threads}), RAM={ctx.hw.ram_gb} GB, GPU={ctx.hw.gpu_name or 'None'}")
print(f"[NB 09] Mode / Seed : RUN_MODE={RUN_MODE}, BATCH_INDEX={BATCH_INDEX}, SEED={RANDOM_SEED}")
print(f"[NB 09] Drive Root  : {ctx.drive_root} (tmp free={ctx.hw.disk_free_tmp_gb} GB)")


In [ ]:
#@title 2. Restore Lightweight CPU Env (`pydantic`) for Master Signoff Aggregation
ensure_cpu_cae_env(ctx)
print("[OK] Master signoff aggregation environment ready.")


In [ ]:
#@title 3. Execute Master Cross-Domain FMEA Signoff & Generate HTML Dashboard + `rev6_patch_orders.json`
worker = ctx.repo_dir / "colab_ultimate_signoff" / "workers" / "nb09_master_signoff_engine.py"
proc = run_in_env(
    profile="cpu_cae",
    script_path=worker,
    args=[
        "--repo-dir", str(ctx.repo_dir),
        "--drive-root", str(ctx.drive_root),
        "--run-mode", RUN_MODE,
        "--git-sha", ctx.git_commit_sha,
        "--seed", str(RANDOM_SEED),
    ],
    cwd=ctx.repo_dir,
    timeout_s=600,
)
print(proc.stdout)


In [ ]:
#@title 4. Strict Schema Validation, Visual Inspection & Read-Only Repo Integrity Proof
from pathlib import Path
from IPython.display import HTML, display

report_path = ctx.reports_dir / "nb09_report.json"
report = validate_report_file(report_path)

print("=" * 84)
print(f"RK T4 DIGITAL TWIN REPORT — NB {report.notebook_id} ({report.notebook_name})")
print(f"Overall Status : {report.overall_status} | Run Mode : {report.run_mode} | Fidelity : {report.model_fidelity}")
print(f"Git Commit SHA : {report.git_commit_sha} | Seed : {report.random_seed}")
print("=" * 84)
for chk in report.checks:
    print(f"[{chk.status:4s}] {chk.check_id:44s} | meas={str(chk.measured_value)[:36]:36s} | lim={chk.limit_value} ({chk.unit})")

print("\n--- Solver Provenance Receipts ---")
for prov in report.solver_provenance:
    bb = prov.model_bbox_mm
    print(
        f"  * {prov.solver_name} ({prov.solver_version}) | scope={prov.model_scope} "
        f"| nodes={prov.mesh_nodes}, elems={prov.mesh_elements} "
        f"| bbox={bb.span_x_mm:.1f}x{bb.span_y_mm:.1f}x{bb.span_z_mm:.1f}mm "
        f"| wall={prov.solve_wall_time_s:.2f}s | files={len(prov.raw_output_files)}"
    )

html_path = Path(report.handoff_artifacts["master_html_report"])
if html_path.is_file():
    print(f"Master Interactive HTML Dashboard saved to: {html_path}")
    print(f"Rev 6 Patch Orders saved to              : {report.handoff_artifacts['rev6_patch_orders_json']}")
    print(f"EVT Tapeout Certificate saved to         : {report.handoff_artifacts['tapeout_certificate_md']}")
    display(HTML(html_path.read_text(encoding="utf-8")))

# MANDATORY Read-Only Invariance Check (tracked design files must be 100% untouched!)
assert verify_repo_unmodified(ctx.repo_dir), "Repository tracked design files were modified!"
print("\n[READ-ONLY VERIFIED] git diff --quiet HEAD -- hardware/ firmware/ src/ optics/ mechanics/ PASSED (0 changes).")
